# 01 — Gemini API Proxy (без GPU)

Самый простой и надёжный вариант. Проксирует запросы к Google Gemini API через OpenAI-совместимый endpoint.

**Требования:** Google AI Studio API ключ ([получить здесь](https://aistudio.google.com/apikey))  
**GPU:** Не нужен  
**Модели:** gemini-2.5-flash (быстрая), gemini-2.5-pro (мощная)  
**Free tier:** ~250 запросов/день для flash

In [ ]:
!pip install -q fastapi uvicorn google-generativeai openai

In [ ]:
import os
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✓ API ключ загружен из Colab Secrets")
except:
    import getpass
    GEMINI_API_KEY = getpass.getpass("Введите Gemini API ключ: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# Настройки
MODEL_NAME = "gemini-2.5-flash"  # или "gemini-2.5-pro"
PORT = 8080

In [ ]:
import google.generativeai as genai
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn
import threading
import time
import uuid

genai.configure(api_key=GEMINI_API_KEY)

app = FastAPI(title="Gemini → OpenAI Proxy")

@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_NAME, "backend": "gemini-api"}

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    messages = body.get("messages", [])
    model = body.get("model", MODEL_NAME)
    temperature = body.get("temperature", 0.0)
    max_tokens = body.get("max_tokens", 200)

    # Map model aliases
    if model in ("local", "default", "gpt-3.5-turbo", "gpt-4"):
        model = MODEL_NAME

    # Extract system instruction and conversation
    system_instruction = None
    conversation = []
    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        if role == "system":
            system_instruction = content
        elif role == "assistant":
            conversation.append({"role": "model", "parts": [content]})
        else:
            conversation.append({"role": "user", "parts": [content]})

    # Ensure conversation is not empty
    if not conversation:
        conversation = [{"role": "user", "parts": ["Hello"]}]

    try:
        gen_config = genai.types.GenerationConfig(
            temperature=temperature,
            max_output_tokens=max_tokens,
        )

        gen_model = genai.GenerativeModel(
            model_name=model,
            system_instruction=system_instruction,
            generation_config=gen_config,
        )

        response = gen_model.generate_content(conversation)
        content = response.text

        return JSONResponse({
            "id": f"chatcmpl-{uuid.uuid4().hex[:8]}",
            "object": "chat.completion",
            "model": model,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": content},
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": 0,
                "completion_tokens": 0,
                "total_tokens": 0
            }
        })
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": {"message": str(e), "type": "server_error"}}
        )

# Start server in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)
print(f"✓ Сервер запущен на порту {PORT}")

In [ ]:
# Установка cloudflared и запуск туннеля
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, re, time

process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

tunnel_url = None
for _ in range(30):
    line = process.stderr.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

if tunnel_url:
    print(f"✓ Туннель активен!")
    print(f"\n{'='*60}")
    print(f"  URL: {tunnel_url}")
    print(f"  Health: {tunnel_url}/health")
    print(f"  API: {tunnel_url}/v1/chat/completions")
    print(f"{'='*60}")
    print(f"\nДля InvoiceLLM config.yaml:")
    print(f'  host: "{tunnel_url.replace("https://", "")}"')
    print(f'  port: 443')
    print(f'  ssl: true')
else:
    print("✗ Не удалось получить URL туннеля")
    print("Попробуйте перезапустить ячейку или используйте ngrok")

In [ ]:
# Тест — отправка invoice-текста
from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="not-needed")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": """Extract from the invoice text: country, doc_type (electricity/telecom/bank/water/gas/tax/other), company, year.
Respond ONLY with JSON: {"country": "...", "doc_type": "...", "company": "...", "year": ...}"""},
        {"role": "user", "content": """RECHNUNG
Wiener Netze GmbH
Erdbergstraße 236, 1110 Wien
Rechnungsdatum: 15.03.2024
Stromrechnung für den Zeitraum 01.01.2024 - 31.03.2024
Gesamtbetrag: EUR 245,67 inkl. MwSt
UID: ATU12345678"""}
    ],
    temperature=0.0,
    max_tokens=200
)

print("Ответ модели:")
print(response.choices[0].message.content)

## Подключение к InvoiceLLM

Добавьте в `config.yaml`:

```yaml
llm:
  servers:
    - name: "Colab Gemini Proxy"
      host: "YOUR-URL.trycloudflare.com"
      port: 443
      ssl: true
      priority: 1
```

Затем запустите: `python run.py classify invoice.pdf`